# 파트 2. 문서 처리 파이프라인

파트 1에서는 문서 253건의 벡터 인덱스를 만들고 고치는 방법을 다뤘습니다. 그 문서는 이미 정리된 Markdown 파일이었습니다. 실제 사내 문서는 PDF, 워드, 엑셀로 옵니다. 머리말과 쪽번호가 본문에 섞여 있고, 표는 칸마다 줄이 바뀌고, 엑셀에는 열 이름 위에 장식 줄이 있습니다. 이 파트에서는 그런 원본 문서에서 검색할 수 있는 청크를 만드는 일을 다룹니다.

이 파트를 마치면 다음을 할 수 있습니다.

1. 문서 처리를 파싱, 정리, 청킹, 메타데이터 붙이기의 네 단계로 나누고, 각 단계에서 검색이 나빠지는 원인을 말합니다.
2. PDF, DOCX, XLSX를 알맞은 도구로 열고, 형식마다 생기는 문제(PDF의 머리말과 표, XLSX의 장식 줄)를 실행해서 확인합니다.
3. 여러 문서에 반복되는 줄을 찾아 지우는 규칙을 만들고, 이 방법이 틀리는 경우를 설명합니다.
4. 표를 통째로 임베딩하면 최대 입력 길이를 넘는 뒷부분이 벡터에 반영되지 않는 것을 확인하고, 행을 한 문장으로 풀어 해결합니다.
5. 문서를 글자 수, 문장, 문서 구조로 자르고 중첩을 주어 청크 수가 어떻게 변하는지 확인하며, 청크 크기를 질문 세트로 측정해 고릅니다. 질문이 적으면 결론이 뒤집힐 수 있다는 점을 설명합니다.
6. 청크에 메타데이터를 붙여 검색 범위를 줄이고, 열리지 않는 파일을 실패 목록에 남기는 파이프라인 함수를 작성합니다.

예제의 실행 결과는 2026-11-26에 실행한 값이고, 학습 노트북을 실행한 출력을 그대로 실었습니다. 이 파트의 예제는 생성 모델 API를 호출하지 않고 임베딩 모델만 PC에서 실행합니다. 노트북의 "준비 (2)" 셀 끝에 있는 연결 확인만 API를 한 번 부릅니다. 검색 점수와 순위는 같은 임베딩 모델과 같은 라이브러리 버전에서 거의 같은 값이 나옵니다. PDF의 글자 수와 줄 수는 PDF를 여는 도구에 따라 다릅니다.

## 이 파트에서 만드는 것

> **요점**
>
> PDF, DOCX, XLSX, MD 문서를 열어 반복 줄을 지우고 청크로 자른 뒤 메타데이터를 붙이는 함수를 만듭니다. 문서 묶음의 폴더를 넣으면 청크 목록과 처리 보고가 나옵니다.

재료는 가상의 반도체 공장의 원본 문서입니다. 파트 1의 문서 253건은 이미 정리한 Markdown이었고, 이 파트의 원본은 문서를 만든 그대로의 파일입니다.

| 폴더 | 형식과 건수 | 내용 |
| --- | --- | --- |
| `data/3일차/docs_raw/` | PDF 8건, DOCX 10건, XLSX 2건 | 예제에서 씁니다. SOP 8건과 주간 회의록 10건, 공정 스펙과 장비 목록 엑셀 2건입니다. SOP와 회의록 18건은 같은 문서의 정리한 Markdown이 `data/2일차/corpus/`에 있습니다 |
| `data/3일차/docs_raw_m2/` | PDF 4건, DOCX 4건, XLSX 1건, MD 1건 | 실습 2에서 씁니다. 예제의 문서와 반복 줄의 종류가 다릅니다 |
| `data/3일차/docs_raw_m2_ext/` | 2건 | 열리지 않는 파일입니다. 파일 하나는 0바이트이고 다른 하나는 텍스트 파일에 `.docx` 확장자를 붙인 것입니다 |

PDF 한 건에서 글을 뽑으면 본문 사이에 머리말, 바닥글, 쪽번호가 섞여 나옵니다.

```
SL-FAB 반도체 · 사내 일반 · 무단 반출 금지
문서번호 SOP-ETCH-007
본 문서의 최신본 여부는 문서관리시스템에서 확인하십시오.
- 1 -
SOP-ETCH-007 에치 장비 ESC 온도 관리 기준
…
```

이 파트가 만드는 것은 이런 원본을 청크 목록으로 바꾸는 함수입니다. 청크 하나는 본문과 메타데이터를 가진 dict입니다. 함수는 청크 목록과 함께, 몇 건을 열었고 몇 줄을 지웠고 어느 파일이 열리지 않았는지를 적은 처리 보고를 돌려줍니다.

처리 보고와 청크 하나의 예입니다. 청크의 본문은 줄였습니다.

```
{'열린 파일': 20, '지운 줄': 32, '실패': []}
{'doc_id': 'MTG-2025-W01', 'type': 'MTG', 'version': 'v1.0', 'dept': '-', 'source': 'MTG-2025-W01.docx',
 'text': 'MTG-2025-W01 장비팀 주간회의록 (2025-08-02)\n참석: 파트장 1, 설비 엔지니어 5, 품질 담당 1\n…'}
```

이 파트는 다음 순서로 진행합니다.

1. 문서 처리의 네 단계를 보고 파이프라인의 구조를 그립니다.
2. PDF, DOCX, XLSX를 열어 형식마다 생기는 문제를 확인합니다.
3. 반복되는 줄을 찾아 지웁니다.
4. 표를 처리합니다.
5. 청크로 자르고, 크기와 중첩을 질문 세트로 측정합니다.
6. 청크에 메타데이터를 붙입니다.
7. 이 모두를 함수 하나로 묶고 실패 목록을 남깁니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

일자마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 일자의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

# 설치: python -m pip install sentence-transformers scikit-learn pypdf python-docx openpyxl pandas
import numpy as np

import embed

DATA = 'data/3일차'
CORPUS = 'data/2일차/corpus'      # 2일차와 같은 문서 253건
OUT = 'out'                        # 예제가 만드는 파일을 두는 폴더
os.makedirs(OUT, exist_ok=True)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 문서 처리의 네 단계

> **요점**
>
> 문서 처리는 파싱, 정리, 청킹, 메타데이터 붙이기의 네 단계입니다. 임베딩 모델은 이 단계가 만든 글만 보므로, 여기서 검색의 품질이 먼저 정해집니다.

임베딩 모델은 원본 파일을 보지 못하고 청크의 글만 받습니다. 청크에 머리말이 섞여 있거나 표의 뒷부분이 빠져 있으면 모델은 그 상태로 벡터를 만듭니다. 검색 결과가 이상할 때 모델이나 `k`를 먼저 바꾸기 쉽지만, 원인이 이 단계에 있을 수 있습니다. 이 파트에서 원인이 될 만한 것을 하나씩 실행해 확인합니다.

문서 처리는 네 단계로 이루어집니다.

| 단계 | 하는 일 | 이 파트에서 다루는 곳 |
| --- | --- | --- |
| 파싱 | 원본 파일에서 글을 뽑습니다. 형식마다 알맞은 도구가 다릅니다 | "형식마다 다른 파싱"(예제 2-1) |
| 정리 | 본문이 아닌 반복 줄을 지웁니다. 표를 문장으로 푸는 일은 XLSX를 파싱할 때 합니다 | "반복 줄 지우기", "표 처리"(예제 2-2, 2-3) |
| 청킹 | 글을 검색 단위로 자릅니다 | "청크 분할", "청크 크기를 측정으로 정하기"(예제 2-4, 2-5) |
| 메타데이터 붙이기 | 청크에 문서 번호, 유형, 버전, 부서를 붙입니다 | "메타데이터"(예제 2-6) |

청크(chunk)는 문서를 검색 단위로 자른 조각이고, 벡터 인덱스의 행 하나가 청크 하나입니다. 문서를 청크로 자르는 일이 청킹(chunking)입니다. 파트 1의 인덱스는 문서 한 건이 청크 하나였습니다.

<img alt="PDF, DOCX, XLSX, MD 네 형식의 원본이 형식별 파서로 들어가 정리, 청킹, 메타데이터 붙이기를 차례로 지나 청크 목록이 되고, 파서와 정리 단계에서 나온 열린 파일 수, 지운 줄 수, 실패 목록이 처리 보고로 따로 남는 그림" width="680" src="">

*청록색 상자가 처리 단계이고, 주황색 상자가 청크 목록, 회색 점선 상자가 처리 보고입니다. 파서는 형식마다 하나씩 네 개이고, 결과는 청크 목록 하나로 모입니다. 청킹과 메타데이터는 형식과 관계없이 같은 코드를 쓰고, 정리 단계의 반복 줄 삭제 규칙은 엑셀을 뺀 형식에 적용합니다("파이프라인으로 묶기" 절). 주황색 점선은 파싱 단계에서 열리지 않은 파일과 정리 단계에서 지운 줄이 처리 보고에 기록되는 경로입니다.*

그림처럼 형식마다 파서를 따로 두고 결과는 같은 청크 목록으로 모으면, 새 형식이 생겨도 파서 하나만 더하면 됩니다. 문서는 계속 개정되고 추가되므로 이 일은 한 번 하고 끝나지 않습니다. 사람이 파일을 손으로 정리하면 다음 달에 다시 해야 합니다. 정리 규칙을 코드로 만들어 두면 문서 묶음이 바뀔 때마다 같은 코드를 다시 실행합니다.

열리지 않은 파일을 알려 주는 것은 처리 보고뿐입니다. 처리 보고는 "파이프라인으로 묶기" 절에서 만듭니다.

## 형식마다 다른 파싱

> **요점**
>
> PDF는 글자의 위치만 있고 제목과 본문의 구분이 없습니다. DOCX는 문단마다 스타일이 있어 제목을 알 수 있고, XLSX는 열 이름 위에 장식 줄이 붙어 있습니다. 형식마다 도구와 읽는 방법이 다릅니다.

파싱은 원본 파일에서 글을 뽑는 일입니다. 한 도구로 모든 형식을 열 수 없어서 형식마다 도구를 고릅니다. 아래 도구는 모두 파이썬 라이브러리이고, 3일차 "실습 환경 준비"의 명령으로 설치합니다.

- **PDF**는 인쇄를 위한 형식입니다. 파일에는 글자와 그 위치만 있고 "이 줄이 제목"이라는 표시가 없습니다. 머리말과 바닥글이 본문과 같은 줄로 섞여 나오고, 표는 칸마다 줄이 바뀝니다. `pypdf`로 엽니다.
- **DOCX**는 편집을 위한 형식입니다. 문단마다 스타일(`Heading 1`, `Normal`)이 있어서, 작성자가 제목 스타일을 썼다면 제목과 본문을 구분할 수 있습니다. 굵은 글씨만 쓰고 스타일이 `Normal`뿐인 DOCX는 PDF처럼 제목을 추측해야 합니다. `python-docx`로 엽니다. 원본이 DOCX로 남아 있으면 같은 문서의 PDF 대신 DOCX를 파싱합니다.
- **XLSX**는 표 자체가 본문입니다. 사람이 보라고 만든 엑셀은 열 이름 위에 제목, 출력일, 빈 줄이 붙어 있어서, 첫 줄을 열 이름으로 읽으면 틀립니다. `pandas`의 `read_excel`로 열고, 병합된 제목처럼 셀의 정보를 보려면 `openpyxl`을 씁니다.

스캔한 PDF는 글자가 아니라 이미지여서 글을 뽑으면 0자가 나옵니다. 이런 파일에는 글자를 인식하는 단계(OCR, optical character recognition)가 앞에 하나 더 필요하고, 이 교재에서는 다루지 않습니다. 0자로 나온 파일은 실패 목록에 남깁니다. 이 파트의 `process()`(예제 2-7)는 글자가 없는 파일을 `ValueError`로 처리합니다.

**예제 2-1. PDF, DOCX, XLSX 열기**

In [ ]:
# 예제 2-1
from docx import Document
from openpyxl import load_workbook
from pypdf import PdfReader

RAW = f'{DATA}/docs_raw'
for kind in ['pdf', 'docx', 'xlsx']:
    files = sorted(os.listdir(f'{RAW}/{kind}'))
    print(f'{kind} {len(files)}건,', files[:2], '…')

pdf 8건, ['SOP-CMP-021.pdf', 'SOP-CVD-009.pdf'] …
docx 10건, ['MTG-2025-W01.docx', 'MTG-2025-W02.docx'] …
xlsx 2건, ['공정스펙_관리한계_일람.xlsx', '장비마스터.xlsx'] …


PDF 한 건을 열어 글을 뽑습니다. 이 문서는 표가 든 SOP입니다.

In [ ]:
# 예제 2-1 (PDF)
PDF = f'{RAW}/pdf/SOP-ETCH-007_v2.1.pdf'
reader = PdfReader(PDF)
plain = '\n'.join(page.extract_text() for page in reader.pages)
line_count = len([line for line in plain.split('\n') if line.strip()])
print(f'{len(reader.pages)}쪽, {len(plain):,}자, {line_count}줄')
print(plain[-210:])

1쪽, 915자, 57줄

2026-06-28
설비1팀
적합
-
ETCH-02
2026-07-02
설비1팀
적합
압력 스파이크 재점검
ETCH-03
2026-07-05
설비2팀
재점검
반사파 버스트 관찰
ETCH-04
2026-07-06
설비2팀
재점검
ESC 온도 상승 추이
ETCH-05
2026-07-10
설비1팀
재점검
RF·압력 동시 변동
ETCH-06
2026-07-11
설비2팀
적합
-


같은 PDF를 열의 위치를 살리는 `pypdf`의 `extraction_mode='layout'` 옵션으로 다시 뽑습니다.

In [ ]:
# 예제 2-1 (PDF, layout 모드)
layout = '\n'.join(page.extract_text(extraction_mode='layout') for page in reader.pages)
rows = [line for line in layout.split('\n') if line.strip()]
print('\n'.join(rows[-8:]))

  호기           최근 점검일         담당          판정          비고
  ETCH-01      2026-06-28     설비1팀        적합          -
  ETCH-02      2026-07-02     설비1팀        적합          압력 스파이크 재점검
  ETCH-03      2026-07-05     설비2팀        재점검         반사파 버스트 관찰
  ETCH-04      2026-07-06     설비2팀        재점검         ESC 온도 상승 추이
  ETCH-05      2026-07-10     설비1팀        재점검         RF·압력 동시 변동
  ETCH-06      2026-07-11     설비2팀        적합          -
본 문서의 최신본 여부는 문서관리시스템에서 확인하십시오.                                                              - 1 -


DOCX는 문단의 스타일로 제목을 알 수 있습니다.

In [ ]:
# 예제 2-1 (DOCX)
document = Document(f'{RAW}/docx/MTG-2025-W01.docx')
paragraphs = [paragraph for paragraph in document.paragraphs if paragraph.text.strip()]
print(f'{len(paragraphs)}문단')
for paragraph in paragraphs:
    if paragraph.style.name.startswith('Heading'):
        print(paragraph.style.name, paragraph.text)

11문단
Heading 1 MTG-2025-W01 장비팀 주간회의록 (2025-08-02)
Heading 2 논의 사항
Heading 2 결정 사항


`공정스펙_관리한계_일람.xlsx`는 첫 줄이 열 이름이 아닙니다. `pandas.read_excel`의 `skiprows` 인자로 앞의 장식 줄을 건너뜁니다.

In [ ]:
# 예제 2-1 (XLSX)
XLSX = f'{RAW}/xlsx/공정스펙_관리한계_일람.xlsx'
sheet = load_workbook(XLSX).active
print([cell.value for cell in sheet[1]][:3])
spec = pd.read_excel(XLSX, skiprows=3)
print(list(spec.columns), f'{len(spec)}행')
master = f'{RAW}/xlsx/장비마스터.xlsx'
print(list(pd.read_excel(master).columns), f'{len(pd.read_excel(master))}행')
print(list(pd.read_excel(master, skiprows=3).columns), f'{len(pd.read_excel(master, skiprows=3))}행(skiprows=3)')

['SL-FAB 공정 스펙 관리 한계 일람', None, None]
['문서번호', '문서명', '적용 장비', '개정일', '버전', '비고'] 30행
['chamber', 'line', 'vendor', 'install_date', 'owner'] 17행
['ETCH-03', 'ETCH', 'Lam', '2022-09-06', '설비1팀'] 14행(skiprows=3)


PDF를 옵션 없이 뽑으면 표의 값이 칸마다 한 줄씩 나옵니다. 예제 2-1 (PDF)의 출력에서 `2026-06-28`, `설비1팀`, `적합`이 한 줄씩 이어지고, 열 이름(최근 점검일, 담당, 판정)이 값 옆에 없어서 `2026-06-28`이 점검일인지 알 수 없습니다. `layout` 모드로 뽑으면 호기마다 값이 한 줄에 모입니다. 이 PDF에서는 `pypdf`가 글머리 기호를 제어 문자(`\x7f`)로 뽑습니다. 검색에 쓸 글에서는 이 문자를 지웁니다(예제 2-2).

DOCX에서는 `Heading 1`이 문서 제목이고 `Heading 2`가 "논의 사항", "결정 사항"입니다. 제목 구조가 파일에 있으므로 `Heading 2` 문단을 경계로 자르면 되고, 제목을 추측하는 규칙은 필요 없습니다. PDF에서는 글자 크기나 위치로 제목을 추측해야 합니다.

`공정스펙_관리한계_일람.xlsx`의 첫 줄은 병합된 제목이어서 `load_workbook`으로 읽으면 첫 칸에만 제목이 있고 나머지 칸은 `None`입니다. 건너뛸 줄 수는 파일마다 다릅니다. `공정스펙_관리한계_일람.xlsx`는 3줄을 건너뛰어야 열 이름(`문서번호`, `문서명`, …)이 나오지만, `장비마스터.xlsx`는 첫 줄이 열 이름이어서 건너뛰지 않습니다. 건너뛸 줄 수를 모든 파일에 같은 값으로 쓰면 두 번째 파일에서 원래의 열 이름 줄과 데이터 2행이 사라지고, 세 번째 데이터 행(`ETCH-03`)이 열 이름이 됩니다. 예제 2-1 (XLSX)의 출력에서 열 이름이 `['ETCH-03', 'ETCH', …]`로 바뀌고 17행이 14행으로 줄어든 것을 볼 수 있습니다. 오류는 나지 않습니다. 이 문제는 실습 2에서 열 이름 줄을 코드로 찾아 해결합니다.

> **보충**
>
> `pypdf` 6.21.2, `python-docx` 1.2.0, `openpyxl` 3.1.5, `pandas` 3.0.6으로 실행했습니다(2026-11-26). PDF의 글자 수와 줄 수는 도구에 따라 다릅니다. 같은 파일을 다른 도구(`pdftotext`)로 열면 줄 수와 줄바꿈 위치가 다르게 나옵니다.

## 반복 줄 지우기

> **요점**
>
> 머리말, 바닥글, 쪽번호는 본문이 아닌데 문서마다 되풀이되는 노이즈입니다. 문서 대부분에 나오는 줄을 세어 후보를 찾을 수 있지만, 같은 양식의 문서에서는 제목과 정형 문장도 후보에 들어오므로 사람이 읽고 지울 줄을 정합니다.

노이즈는 본문이 아닌데 문서마다 되풀이되는 줄입니다. 머리말, 바닥글, 쪽번호가 여기에 해당합니다. 노이즈가 들어간 청크는 그 글까지 벡터에 반영됩니다. 노이즈는 문서마다 같은 글이므로 청크를 구별하는 정보가 아닙니다.

노이즈를 지우는 규칙은 낱말 목록으로 만듭니다. 줄에 목록의 낱말이 들어 있거나 줄 전체가 `- 1 -` 꼴의 쪽번호이면 노이즈로 봅니다. 쪽번호 꼴은 정규식(regular expression, 문자열의 패턴을 기호로 적는 표기법)으로 적습니다. 낱말이 들어 있는 줄은 모두 지우므로, 본문 줄에 같은 낱말이 있으면 규칙이 본문 줄도 지웁니다("파이프라인으로 묶기" 절의 엑셀 사례).

**예제 2-2. 반복 줄을 찾아 지우기**

In [ ]:
# 예제 2-2
import re

DROP_WORDS = ['무단 반출 금지', '문서관리시스템', '문서번호']
PAGE_NUMBER = re.compile(r'-\s*\d+\s*-')


def is_noise(line, words=DROP_WORDS):
    """줄에 노이즈 낱말이 들어 있거나 쪽번호 줄이면 True. 공백을 뺀 글로 비교합니다."""
    squeezed = ''.join(line.split())
    return (any(''.join(word.split()) in squeezed for word in words)
            or bool(PAGE_NUMBER.fullmatch(line.strip())))


lines = [line.strip() for line in plain.split('\n') if line.strip()]
noise = [line for line in lines if is_noise(line)]
print(f'{len(noise)}/{len(lines)}줄, {len(noise) / len(lines) * 100:.1f}%')
print(noise)

4/57줄, 7.0%
['SL-FAB 반도체 · 사내 일반 · 무단 반출 금지', '문서번호 SOP-ETCH-007', '본 문서의 최신본 여부는 문서관리시스템에서 확인하십시오.', '- 1 -']


문서가 많으면 노이즈 낱말을 눈으로 찾기 어렵습니다. 전체 문서의 80%를 넘는 문서에 똑같이 나오는 줄을 세어 후보를 찾습니다. 한 문서에서 여러 번 나와도 한 번만 셉니다.

In [ ]:
# 예제 2-2 (반복 줄 후보)
from collections import Counter


def repeated_lines(line_lists, ratio=0.8):
    """line_lists는 문서마다의 줄 목록입니다. ratio를 넘는 문서에 나오는 줄을 돌려줍니다."""
    counter = Counter(line for lines in line_lists for line in set(lines))
    return sorted(line for line, count in counter.items() if count > len(line_lists) * ratio)


pdf_lines = []
for name in sorted(os.listdir(f'{RAW}/pdf')):
    text = '\n'.join(page.extract_text() for page in PdfReader(f'{RAW}/pdf/{name}').pages)
    pdf_lines.append([line.strip() for line in text.split('\n') if line.strip()])
print(f'PDF {len(pdf_lines)}건:', repeated_lines(pdf_lines))

docx_lines = []
for name in sorted(os.listdir(f'{RAW}/docx')):
    paragraphs = Document(f'{RAW}/docx/{name}').paragraphs
    docx_lines.append([paragraph.text.strip() for paragraph in paragraphs if paragraph.text.strip()])
print(f'DOCX {len(docx_lines)}건:', repeated_lines(docx_lines))

PDF 8건: ['- 1 -', 'SL-FAB 반도체 · 사내 일반 · 무단 반출 금지', '\x7f 등급: 사내 일반', '본 문서의 최신본 여부는 문서관리시스템에서 확인하십시오.']
DOCX 10건: ['결정 사항', '논의 사항', '등급: 사내 일반', '안건별 담당자는 회의 종료 후 파트장이 개별 통지했습니다.']


노이즈로 판정한 줄을 지우고, `pypdf`가 글머리 기호로 뽑은 제어 문자도 지웁니다.

In [ ]:
# 예제 2-2 (지우기)
CONTROL = re.compile(r'[\x00-\x08\x0b-\x1f\x7f]')
cleaned = [CONTROL.sub('', line).strip() for line in lines if not is_noise(line)]
print(f'{len(lines)}줄 → {len(cleaned)}줄')
print(repr(lines[5]), '→', repr(cleaned[1]))

57줄 → 53줄
'\x7f 적용 장비: ETCH-01 ~ ETCH-06 전 호기' → '적용 장비: ETCH-01 ~ ETCH-06 전 호기'


반복 줄이 항상 같은 글자로 나오지는 않습니다. 실습 2의 문서에서 "기밀" 줄을 뽑으면 글자 사이가 벌어져 있습니다.

In [ ]:
# 예제 2-2 (글자 사이가 벌어진 줄)
text = '\n'.join(page.extract_text() for page in PdfReader(f'{DATA}/docs_raw_m2/pdf/NOTI-031.pdf').pages)
line = [line for line in text.split('\n') if '밀' in line][0]
print(repr(line))
print("'기밀' in line:", '기밀' in line, end=', ')
print('is_noise(line, ["기밀"]):', is_noise(line, ['기밀']))

'기 밀  ·  내 부  검 토 용  초 안  ·  배 포  금 지'
'기밀' in line: False, is_noise(line, ["기밀"]): True


예제 2-2의 첫 출력에서 이 PDF는 57줄 중 4줄이 노이즈이고 비율은 7.0%입니다. 이 PDF는 1쪽이라 머리말, 바닥글, 쪽번호가 한 번씩 나옵니다. 쪽 수가 늘면 쪽마다 되풀이되어 지울 줄도 늘어납니다. 지운 뒤에는 57줄이 53줄이 됩니다. 낱말 목록의 `문서번호`는 문서마다 번호가 달라서 아래 반복 줄 후보에는 나오지 않고, PDF 한 건을 직접 읽고 더한 낱말입니다.

반복 줄 후보는 두 묶음에서 다르게 나옵니다. PDF 8건에서 후보 4줄은 바닥글, 머리말, 쪽번호, 그리고 "등급: 사내 일반" 줄입니다. 이 가운데 바닥글, 머리말, 쪽번호는 지울 줄입니다. DOCX 10건에서도 후보가 4줄 나오지만 그중 "논의 사항", "결정 사항"은 회의록의 절 제목이고 "안건별 담당자는 회의 종료 후 파트장이 개별 통지했습니다."는 본문 문장입니다. 같은 양식의 회의록은 제목과 정형 문장이 문서마다 똑같이 나오므로 후보에 들어옵니다. 후보를 그대로 지우면 절 제목이 사라집니다. 반복 줄 탐지는 후보를 찾는 데만 쓰고, 지우기 전에 사람이 후보를 읽어 확인합니다. "등급: 사내 일반"처럼 노이즈인지 본문인지 애매한 줄은 그 줄의 내용으로 검색하거나 필터링할 일이 있는지로 판단합니다. 이 교재는 이 줄을 문서의 속성으로 보고 지우지 않습니다(예제 2-7의 청크에 남아 있습니다). 등급을 메타데이터로 쓰지 않아서 "메타데이터" 절에서도 따로 뽑지 않습니다.

예제 2-2 (글자 사이가 벌어진 줄)의 출력에서 `'기밀' in line`은 거짓입니다. 글자 사이에 공백이 들어 있기 때문입니다. 공백을 모두 뺀 글로 비교하면(`is_noise`의 `squeezed`) 같은 줄을 찾습니다. 같은 PDF도 도구에 따라 줄이 다르게 나오므로, 규칙은 사용하는 도구로 뽑은 결과를 읽고 만듭니다.

이 노이즈가 검색 결과를 얼마나 해치는지는 예제 2-5에서 청크 크기와 함께 측정합니다.

## 표 처리

> **요점**
>
> 표를 통째로 임베딩하면 최대 입력 길이를 넘는 뒷부분의 행이 벡터에 반영되지 않습니다. 한 행을 한 문장으로 풀면 행마다 벡터가 생겨, 질문이 각 행과 직접 비교됩니다.

표는 RAG에서 다루기 어려운 문서 요소입니다. 사람은 열 이름을 보고 값의 뜻을 알지만, 글로 뽑은 표에서는 열 이름이 값 옆에 없어서 `2026-07-11`이 점검일인지 알 수 없습니다.

표를 다루는 방법은 세 가지입니다.

- 표를 글 그대로 통째로 임베딩합니다.
- 열의 위치를 살려 행마다 한 줄로 뽑습니다(PDF의 `layout` 모드).
- 행마다 열 이름을 붙여 한 문장으로 풉니다.

통째로 임베딩하는 방법에는 한계가 하나 더 있습니다. 임베딩 모델은 최대 입력 길이를 넘는 부분을 잘라 냅니다. 이 파트의 모델은 512토큰입니다. 긴 표를 통째로 넣으면 뒤쪽 행은 벡터에 반영되지 않고, `embed.passages()`는 오류 없이 벡터를 돌려줍니다. 예제 2-3처럼 토크나이저를 직접 호출하면 `Token indices sequence length is longer than the specified maximum sequence length` 경고가 출력됩니다. 교재의 실행 결과에는 이런 경고 출력(stderr)을 싣지 않습니다. 임베딩 결과에는 잘렸다는 표시가 없습니다. 예제 2-3에서 이를 확인합니다.

**예제 2-3. 표를 통째로 넣을 때와 행을 문장으로 풀 때**

In [ ]:
# 예제 2-3
model = embed.load()
blob = spec.to_string(index=False)
token_count = len(model.tokenizer('passage: ' + blob)['input_ids'])
print(f'표 전체 {token_count:,}토큰, 최대 입력 길이 {model.max_seq_length:,}토큰')

full = embed.passages([blob])[0]
for removed in [4, 8, 12, 16]:
    shorter = embed.passages([spec.iloc[:-removed].to_string(index=False)])[0]
    print(f'마지막 {removed:2d}행을 지운 표와의 유사도 {float(full @ shorter):.4f}')

임베딩 모델: intfloat/multilingual-e5-small (PC에 저장된 파일을 씁니다)
표 전체 722토큰, 최대 입력 길이 512토큰
마지막  4행을 지운 표와의 유사도 1.0000
마지막  8행을 지운 표와의 유사도 1.0000
마지막 12행을 지운 표와의 유사도 0.9986
마지막 16행을 지운 표와의 유사도 0.9955


행 하나를 한 문장으로 풀고 검색해 봅니다.

In [ ]:
# 예제 2-3 (행을 한 문장으로)
def row_sentence(row):
    return (f"{row['문서번호']}: {row['적용 장비']} 적용, 문서명 {row['문서명']}, "
            f"개정일 {row['개정일']}, 버전 {row['버전']}")


sentences = [row_sentence(row) for _, row in spec.iterrows()]
print(f'{len(sentences)}문장, 평균 {round(sum(map(len, sentences)) / len(sentences))}자')
print(sentences[1])

query = embed.query('SPEC-030 은 언제 개정됐나요?')
scores = embed.passages(sentences) @ query
best = int(scores.argmax())
print('행 문장 1위:', sentences[best], round(float(scores[best]), 3))
print('표 통째로 넣은 벡터:', round(float(full @ query), 3))

30문장, 평균 68자
SPEC-002: ETCH-06 적용, 문서명 RCP-ETCH-693 관리 한계, 개정일 2026-07-17, 버전 v1.0
행 문장 1위: SPEC-030: ETCH-05 적용, 문서명 RCP-ETCH-154 관리 한계, 개정일 2025-07-11, 버전 v1.0 0.875
표 통째로 넣은 벡터: 0.824


표 전체는 722토큰이어서 최대 입력 길이 512토큰을 넘습니다. 마지막 8행을 지워도 벡터가 바뀌지 않았고(유사도 1.0000), 12행을 지우자 처음으로 0.9986이 됩니다. 뒤쪽 행은 벡터에 반영되지 않았습니다. 그래서 표 끝에 있는 "SPEC-030" 행의 내용은 표 벡터에 들어가지 않습니다.

행을 한 문장으로 풀면 문장이 30개가 되고 평균 68자입니다. 열 이름을 문장에 넣었으므로 `2025-07-11` 같은 값이 개정일이라는 것이 글에 남습니다. 질문 "SPEC-030 은 언제 개정됐나요?"의 1위는 SPEC-030의 행 문장(유사도 0.875)입니다. 표를 통째로 넣은 벡터와 질문의 유사도는 0.824로 더 낮습니다(행 문장 30개 가운데 1위와, 표 하나의 벡터를 비교한 값입니다).

표를 문장으로 풀어 넣는 일은 세 방법 가운데 작업이 가장 많습니다. 그 대신 질문한 행이 따로 벡터를 가지므로 다른 행과 순위를 비교할 수 있고, 이 예제에서 SPEC-030의 행 문장은 30개 가운데 1위였습니다. 표 벡터에는 이 행의 내용이 들어 있지 않습니다. 검색 결과는 후보 사이의 순위로 정해지므로, 0.875와 0.824라는 점수의 차이보다 이 순위를 봅니다. 장비 목록과 스펙표, 알람 코드표처럼 표가 핵심인 문서는 이 방법을 권합니다. 세 방법 가운데 `layout` 모드는 검색에서 비교하지 않았습니다.

## 청크 분할

> **요점**
>
> 문서를 청크로 자르는 기준은 글자 수, 문장, 문서 구조입니다. 글자 수로 자르면 문장 한가운데가 끊기고, 문서 구조로 자르면 청크가 어느 문서의 어느 절인지 알 수 있습니다. 중첩을 주면 이웃한 청크가 겹치고, 문서가 청크보다 여러 배 길면 청크 수가 늘어납니다.

임베딩 모델은 최대 입력 길이를 넘는 부분을 잘라 내고, 검색은 답이 있는 대목을 찾아야 합니다. 그래서 긴 문서는 청크로 잘라 청크마다 벡터를 만듭니다. 청크 크기에는 상충 관계가 있습니다. 너무 작으면 문맥이 끊겨 청크만 봐서는 무엇에 대한 글인지 알 수 없고, 너무 크면 한 벡터에 여러 주제가 섞여 질문과 관련된 부분의 비중이 줄어 유사도가 낮아지고 프롬프트에 넣는 토큰도 늘어납니다. 이 교재의 문서는 짧아서(127~603자, 중앙값 272자) 길이 때문에 자를 필요는 거의 없습니다. 그래도 자르는 방법이 검색 결과를 바꾸는지 보기 위해 여러 방법을 비교합니다.

자르는 방법은 세 가지입니다.

- **글자 수**(fixed-size chunking): 정한 글자 수마다 자릅니다. 가장 단순하고, 문장 한가운데에서 끊길 수 있습니다.
- **문장**(sentence-based chunking): 마침표나 줄바꿈에서 끊고, 정한 길이에 가깝게 문장을 이어 붙입니다.
- **문서 구조**(structure-based chunking): `##`로 시작하는 절 단위로 자르고, 청크 앞에 문서 제목을 붙입니다. 청크만 봐도 어느 문서의 어느 절인지 알 수 있습니다.

문서 구조로 자르려면 문서에 구조가 있어야 합니다. Markdown의 `##`와 DOCX의 `Heading 2`는 구조이고, PDF는 구조가 없어 글자 수나 문장으로 자릅니다.

중첩(overlap)은 앞 청크의 끝부분을 다음 청크의 앞에도 넣어 이웃한 청크가 겹치게 하는 것입니다. 경계에 걸친 문장이 두 청크 어느 쪽에도 온전히 들어가지 않는 문제를 줄입니다. 중첩에는 비용이 있습니다. 청크 수가 늘고 그만큼 인덱스가 커지며 검색할 벡터가 늘어납니다.

**예제 2-4. 청크로 자르는 세 방법과 중첩**

In [ ]:
# 예제 2-4
names = sorted(file[:-3] for file in os.listdir(CORPUS) if file.endswith('.md'))
docs = [open(f'{CORPUS}/{name}.md', encoding='utf-8').read() for name in names]
lengths = [len(doc) for doc in docs]
print(f'{len(docs)}건, 글자 수 최소 {min(lengths):,}자, 중앙 {int(np.median(lengths)):,}자, 최대 {max(lengths):,}자')


def chunk_fixed(text, size=300, overlap=0):
    """size 글자씩 자릅니다. overlap 글자는 다음 청크의 앞에도 넣습니다."""
    step = size - overlap
    # 시작 위치는 len(text) - overlap 앞까지만 둡니다. 남은 글이 모두 앞 청크의 중첩 부분이면 새 청크를 만들지 않습니다
    return [text[start:start + size] for start in range(0, max(len(text) - overlap, 1), step)
            if text[start:start + size].strip()]


def chunk_sentences(text, size=300):
    """마침표나 줄바꿈에서 끊고, 길이가 size에 가깝게 문장을 이어 붙입니다."""
    pieces = re.split(r'(?<=[.!?])\s+|\n+', text)
    chunks, current = [], ''
    for piece in pieces:
        if current and len(current) + len(piece) > size:
            chunks.append(current.strip())
            current = ''
        current += piece + ' '
    if current.strip():
        chunks.append(current.strip())
    return chunks


def chunk_sections(text):
    """## 절 단위로 자르고, 청크 앞에 문서 제목을 붙입니다."""
    title = text.split('\n', 1)[0].lstrip('# ').strip()
    parts, current, heading = [], [], None
    for line in text.split('\n'):
        if line.startswith('## '):
            if current:
                parts.append((heading, '\n'.join(current)))
            heading, current = line[3:].strip(), []
        else:
            current.append(line)
    if current:
        parts.append((heading, '\n'.join(current)))
    return [f'[{title}] {heading or ""}\n{body}'.strip() for heading, body in parts if body.strip()]

253건, 글자 수 최소 127자, 중앙 272자, 최대 603자


SOP 한 건을 100자로 잘라 경계를 봅니다. 문장 단위와 문서 구조 단위로 자른 결과도 봅니다.

In [ ]:
# 예제 2-4 (잘린 곳)
target = docs[names.index('SOP-ETCH-007_v2.1')]
pieces = chunk_fixed(target, 100)
print(f'{len(pieces)}조각, 경계:', repr(pieces[1][-24:]), '|', repr(pieces[2][:24]))
print('문장 단위 길이:', [len(piece) for piece in chunk_sentences(target)])
print(chunk_sections(target)[1][:70])

7조각, 경계: '~ 62.0°C** 입니다.\n- **상한 6' | '2.0°C 를 5분 이상 지속 초과하면 챔버'
문장 단위 길이: [273, 296, 27]
[SOP-ETCH-007 에치 장비 ESC 온도 관리 기준] 관리 기준
- ESC(정전척) 온도의 정상 운전 범위는 **58.


방법마다 253건에서 청크가 몇 개 나오는지, 중첩을 주면 얼마나 늘어나는지 셉니다.

In [ ]:
# 예제 2-4 (청크 수)
for label, split in [('글자 100', lambda text: chunk_fixed(text, 100)), ('글자 200', lambda text: chunk_fixed(text, 200)),
                     ('글자 400', lambda text: chunk_fixed(text, 400)), ('문장 300', chunk_sentences),
                     ('문서 구조', chunk_sections), ('분할 없음', lambda text: [text])]:
    print(f'{label:8s} 청크 {sum(len(split(doc)) for doc in docs):,}건')

base = sum(len(chunk_fixed(doc, 400)) for doc in docs)
for overlap in [0, 40, 80, 160]:
    count = sum(len(chunk_fixed(doc, 400, overlap)) for doc in docs)
    print(f'400자, 중첩 {overlap:3d}자 → {count:,}건({count / base:.2f}배)')

글자 100   청크 825건
글자 200   청크 469건
글자 400   청크 282건
문장 300   청크 356건
문서 구조    청크 757건
분할 없음    청크 253건
400자, 중첩   0자 → 282건(1.00배)
400자, 중첩  40자 → 282건(1.00배)
400자, 중첩  80자 → 282건(1.00배)
400자, 중첩 160자 → 282건(1.00배)


글자 수로 자른 결과의 경계는 `'~ 62.0°C** 입니다.\n- **상한 6'`에서 끊겨 다음 청크가 `'2.0°C 를 5분 이상…'`으로 시작합니다. 숫자 62.0이 두 청크로 갈라져서, 어느 쪽도 "상한 62.0°C"라는 뜻을 온전히 갖지 못합니다. 이 SOP는 문장 단위로 자르면 청크 3개(길이 273, 296, 27자)가 됩니다. 마침표나 줄바꿈에서 끊으므로 문장이 잘리지 않습니다. 문서 구조 단위로 자른 청크는 `[SOP-ETCH-007 에치 장비 ESC 온도 관리 기준] 관리 기준`으로 시작해, 청크만 봐도 어느 문서의 어느 절인지 알 수 있습니다.

253건을 방법별로 자르면 글자 100자는 825건, 200자는 469건, 400자는 282건이고 문장 300자는 356건, 문서 구조는 757건입니다. 분할 없음(문서 한 건을 통째로 청크 하나로 둔 경우)은 253건입니다. 400자에 중첩 40자, 80자, 160자를 주어도 청크는 282건 그대로입니다. 이 코퍼스에서 가장 긴 문서가 603자여서 400자로 자르면 청크가 많아야 두 개입니다. 중첩을 160자까지 주어도 두 번째 청크의 시작 위치가 앞으로 당겨질 뿐 세 번째 청크는 생기지 않습니다. 중첩이 청크 수를 늘리는 것은 문서가 청크 크기보다 여러 배 길어 경계가 많을 때입니다. 200자 청크에서 측정한 결과는 예제 2-5에 있습니다.

`chunk_fixed`는 앞 청크의 끝이 문서 끝에 닿으면 다음 청크를 만들지 않습니다(`range`의 끝을 `len(text) - overlap`으로 둔 부분). 이 조건이 없으면 문서 끝에서 앞 청크에 이미 다 들어 있는 조각을 한 번 더 만들어, 중첩 때문이 아닌 청크가 늘어납니다.

## 청크 크기를 측정으로 정하기

> **요점**
>
> 청크 크기는 블로그의 권장값이 아니라 질문 세트로 측정해 정합니다. 질문 세트가 달라지면 크기의 순서가 달라지므로, 이 코퍼스에서는 청크 크기에 따른 차이를 말할 수 없습니다.

"청크는 500토큰, 중첩 50토큰" 같은 권장값은 누군가의 문서에서 나온 결과입니다. 내 문서에도 맞는지는 알 수 없습니다. 청크 크기와 중첩은 바꿀 수 있는 설정이고, 바꾼 설정이 좋아졌는지는 같은 질문 세트로 측정해야 알 수 있습니다. 질문 세트는 질문마다 정답 문서를 사람이 정해 둔 것입니다. 질문은 실제로 받은 문의나 인수인계 문서의 Q&A에서 모읍니다.

청크로 자르면 검색 결과도 청크 단위가 되므로, 측정 방법도 청크 단위로 바꿉니다. 점수가 높은 청크 순서로 정렬해 다음 두 값을 구합니다. 2일차 파트 3의 recall@k를 청크 단위로 센 것입니다. 정답 문서의 청크가 상위 k개 안에 있으면 맞힌 것으로 셉니다.

- **recall@1**: 1위 청크가 정답 문서의 청크인 질문의 비율입니다.
- **recall@3**: 상위 3개 청크 가운데 정답 문서의 청크가 있는 질문의 비율입니다.

측정에는 질문 세트 두 가지를 씁니다. 청크 크기 측정용으로 만든 질문 10개(`청크_질문_10.csv`)와 2일차 파트 3의 질문 20개(`검색_질문_20.csv`)입니다.

**예제 2-5. 청크 크기를 질문 세트로 측정하기**

In [ ]:
# 예제 2-5
QUESTIONS_10 = pd.read_csv(f'{DATA}/청크_질문_10.csv')
QUESTIONS_20 = pd.read_csv('data/2일차/검색_질문_20.csv')


def index_chunks(split, texts=docs):
    """문서마다 split으로 자르고 (청크의 벡터 배열, 청크의 문서 번호 목록)을 돌려줍니다."""
    chunks, owners = [], []
    for name, text in zip(names, texts):
        for chunk in split(text):
            chunks.append(chunk)
            owners.append(name)
    return embed.passages(chunks), owners


def measure(vectors, owners, questions):
    """(recall@1 %, recall@3 %, 1위와 4위 청크의 점수 차 평균)을 돌려줍니다."""
    recall1 = recall3 = gap = 0
    for question, gold in zip(questions['질문'], questions['정답']):
        scores = vectors @ embed.query(question)
        order = np.argsort(-scores, kind='stable')
        found = [owners[position] for position in order[:3]]
        recall1 += found[0] == gold
        recall3 += gold in found
        gap += scores[order[0]] - scores[order[3]]
    count = len(questions)
    return recall1 / count * 100, recall3 / count * 100, gap / count


METHODS = [('글자 100', lambda text: chunk_fixed(text, 100)), ('글자 200', lambda text: chunk_fixed(text, 200)),
           ('글자 400', lambda text: chunk_fixed(text, 400)), ('문장 300', chunk_sentences),
           ('문서 구조', chunk_sections), ('분할 없음', lambda text: [text])]
for label, split in METHODS:
    vectors, owners = index_chunks(split)
    row10 = measure(vectors, owners, QUESTIONS_10)
    row20 = measure(vectors, owners, QUESTIONS_20)
    print(f'{label:8s} 청크 {len(owners):4d} | 질문 10개 recall@1 {row10[0]:3.0f}% recall@3 {row10[1]:3.0f}% | '
          f'질문 20개 recall@1 {row20[0]:3.0f}% recall@3 {row20[1]:3.0f}%')

글자 100   청크  825 | 질문 10개 recall@1  60% recall@3  60% | 질문 20개 recall@1  75% recall@3  95%
글자 200   청크  469 | 질문 10개 recall@1  60% recall@3  70% | 질문 20개 recall@1  75% recall@3  85%
글자 400   청크  282 | 질문 10개 recall@1  60% recall@3  80% | 질문 20개 recall@1  80% recall@3  90%
문장 300   청크  356 | 질문 10개 recall@1  60% recall@3  70% | 질문 20개 recall@1  80% recall@3  90%
문서 구조    청크  757 | 질문 10개 recall@1  50% recall@3  70% | 질문 20개 recall@1  80% recall@3  95%
분할 없음    청크  253 | 질문 10개 recall@1  60% recall@3  80% | 질문 20개 recall@1  85% recall@3  90%


중첩을 주었을 때의 recall@3도 같은 방식으로 측정합니다.

In [ ]:
# 예제 2-5 (중첩)
for overlap in [0, 20, 40, 60]:
    vectors, owners = index_chunks(lambda text, overlap=overlap: chunk_fixed(text, 200, overlap))
    row10 = measure(vectors, owners, QUESTIONS_10)
    row20 = measure(vectors, owners, QUESTIONS_20)
    print(f'200자, 중첩 {overlap:2d}자 청크 {len(owners):4d} | 질문 10개 recall@1 {row10[0]:3.0f}% recall@3 {row10[1]:3.0f}% | '
          f'질문 20개 recall@1 {row20[0]:3.0f}% recall@3 {row20[1]:3.0f}%')

200자, 중첩  0자 청크  469 | 질문 10개 recall@1  60% recall@3  70% | 질문 20개 recall@1  75% recall@3  85%
200자, 중첩 20자 청크  481 | 질문 10개 recall@1  60% recall@3  70% | 질문 20개 recall@1  75% recall@3  85%
200자, 중첩 40자 청크  489 | 질문 10개 recall@1  60% recall@3  70% | 질문 20개 recall@1  85% recall@3  85%
200자, 중첩 60자 청크  526 | 질문 10개 recall@1  60% recall@3  80% | 질문 20개 recall@1  80% recall@3  85%


반복 줄이 남아 있을 때 검색이 얼마나 나빠지는지도 측정합니다. 모든 문서 끝에 머리말 한 줄과 바닥글 한 줄을 붙여 같은 측정을 합니다.

In [ ]:
# 예제 2-5 (반복 줄이 남아 있을 때)
REPEATED_LINES = '\n\nSL-FAB 반도체 · 사내 일반 · 무단 반출 금지\n본 문서의 최신본 여부는 문서관리시스템에서 확인하십시오.'
noisy_docs = [doc + REPEATED_LINES for doc in docs]
for label, split in [('글자 100', lambda text: chunk_fixed(text, 100)), ('글자 400', lambda text: chunk_fixed(text, 400)),
                     ('분할 없음', lambda text: [text])]:
    for tag, texts in [('원본', docs), ('반복 줄 추가', noisy_docs)]:
        vectors, owners = index_chunks(split, texts)
        similarity = vectors @ vectors.T
        mean_similarity = (similarity.sum() - len(vectors)) / (len(vectors) * (len(vectors) - 1))
        row20 = measure(vectors, owners, QUESTIONS_20)
        print(f'{label:8s} {tag:8s} 청크 {len(owners):4d}, 청크끼리 유사도 평균 {mean_similarity:.3f}, '
              f'질문 20개 recall@1 {row20[0]:3.0f}% recall@3 {row20[1]:3.0f}%, 1위와 4위의 점수 차 {row20[2]:.4f}')

글자 100   원본       청크  825, 청크끼리 유사도 평균 0.854, 질문 20개 recall@1  75% recall@3  95%, 1위와 4위의 점수 차 0.0160
글자 100   반복 줄 추가  청크  990, 청크끼리 유사도 평균 0.858, 질문 20개 recall@1  75% recall@3  95%, 1위와 4위의 점수 차 0.0160
글자 400   원본       청크  282, 청크끼리 유사도 평균 0.855, 질문 20개 recall@1  80% recall@3  90%, 1위와 4위의 점수 차 0.0230
글자 400   반복 줄 추가  청크  339, 청크끼리 유사도 평균 0.871, 질문 20개 recall@1  90% recall@3  90%, 1위와 4위의 점수 차 0.0225
분할 없음    원본       청크  253, 청크끼리 유사도 평균 0.865, 질문 20개 recall@1  85% recall@3  90%, 1위와 4위의 점수 차 0.0226
분할 없음    반복 줄 추가  청크  253, 청크끼리 유사도 평균 0.892, 질문 20개 recall@1  90% recall@3  90%, 1위와 4위의 점수 차 0.0210


같은 인덱스를 질문 10개와 질문 20개로 측정한 결과는 다음과 같습니다.

| 자르는 방법 | 청크 수 | 질문 10개: recall@1 | 질문 10개: recall@3 | 질문 20개: recall@1 | 질문 20개: recall@3 |
| --- | --- | --- | --- | --- | --- |
| 글자 100 | 825 | 60% | 60% | 75% | 95% |
| 글자 200 | 469 | 60% | 70% | 75% | 85% |
| 글자 400 | 282 | 60% | 80% | 80% | 90% |
| 문장 300 | 356 | 60% | 70% | 80% | 90% |
| 문서 구조 | 757 | 50% | 70% | 80% | 95% |
| 분할 없음 | 253 | 60% | 80% | 85% | 90% |

<img alt="자르는 방법 여섯 가지의 recall@3을 질문 10개와 질문 20개로 측정한 값을 가로 축 위의 점 두 개로 보인 그림. 질문 10개에서는 글자 100이 60%로 가장 낮고 글자 400과 분할 없음이 80%로 가장 높지만, 질문 20개에서는 글자 100과 문서 구조가 95%로 가장 높고 글자 200이 85%로 가장 낮다" width="680" src="">

*주황색 점은 질문 10개, 청록색 점은 질문 20개로 측정한 recall@3입니다. 질문 10개에서는 글자 100(60%)이 가장 낮고 글자 400과 분할 없음(80%)이 가장 높습니다. 질문 20개에서는 글자 100과 문서 구조(95%)가 가장 높고 글자 200(85%)이 가장 낮습니다. 같은 인덱스를 질문 세트만 바꿔 측정했는데 크기의 순서가 바뀝니다.*

질문 10개로 보면 recall@3이 글자 100에서 60%, 200에서 70%, 400에서 80%로 오릅니다. 이 결과만 보면 "청크가 작을수록 나빠진다"고 결론 내리기 쉽습니다. 같은 인덱스를 질문 20개로 측정하면 글자 100의 recall@3이 95%로 가장 높습니다. 순서가 달라졌습니다. 두 질문 세트에서 가장 높은 값과 가장 낮은 값의 차이는 10개일 때 2문항(20%p), 20개일 때 2문항(10%p)입니다. 질문 세트가 달라지면 크기의 순서가 달라지고 차이도 질문 두 개 수준이므로, 이 코퍼스와 이 질문 세트로는 어느 청크 크기가 낫다고 말할 수 없습니다.

이 코퍼스는 문서가 짧습니다(중앙값 272자). 글자 400으로 자르면 문서 대부분이 청크 하나나 둘이 되고, 크기를 바꿔도 청크의 내용이 크게 달라지지 않습니다. 긴 문서가 많은 코퍼스에서는 결과가 다를 수 있습니다. 어느 쪽이든 청크 크기는 내 문서와 내 질문 세트로 측정해 정하고, 질문이 열 개 안팎이면 결론을 내리지 않습니다. 질문 세트를 만드는 일은 RAG 프로젝트의 첫 과제입니다. 질문 세트가 있으면 청크 크기, 임베딩 모델, `k`를 숫자로 비교할 수 있습니다.

200자 청크에 중첩을 주면 청크가 469건에서 481건(20자), 489건(40자), 526건(60자, 1.12배)으로 늘어납니다. recall@3은 질문 10개에서 중첩 60자일 때만 70%에서 80%로 오르는데, 질문 한 개의 차이입니다. 질문 20개에서는 중첩 0, 20, 40, 60자 모두 85%로 같습니다. 이 코퍼스에서는 중첩이 검색을 나아지게 한다고 말할 수 없고, 청크 수와 인덱스 크기가 늘어나는 비용만 확실합니다. 중첩은 기본값으로 켜 두지 않고 질문 세트로 측정한 뒤 정합니다.

모든 문서 끝에 같은 머리말과 바닥글 두 줄을 붙이면 청크끼리의 유사도 평균이 올라갑니다. 분할 없음은 0.865에서 0.892로, 글자 400은 0.855에서 0.871로, 글자 100은 0.854에서 0.858로 올랐습니다. 청크 수도 늘어서, 글자 100은 825건에서 990건이 되고 글자 400은 282건에서 339건이 됩니다. 분할 없음은 문서 한 건이 청크 하나라서 253건 그대로입니다.

검색 결과는 크게 나빠지지 않았습니다. 질문 20개의 recall@3은 세 경우 모두 그대로입니다(95%, 90%, 90%). recall@1은 글자 400에서 80%가 90%로, 분할 없음에서 85%가 90%로 오히려 올랐는데, 질문 한두 개의 변동입니다. 점수 차는 1위 청크의 점수에서 4위 청크의 점수를 뺀 값으로, 1위 청크가 4위 청크보다 점수가 얼마나 높은지 보여 줍니다. 이 값이 작으면 상위 청크들의 점수가 서로 가깝습니다. 이 값은 분할 없음에서 0.0226이 0.0210으로 줄었고 글자 100에서는 변하지 않았습니다. 이 코퍼스에서는 노이즈가 검색 결과를 해친다는 것을 확인하지 못했습니다.

그래도 반복 줄은 지웁니다. 청크 수가 늘어 인덱스와 검색 시간이 늘고, 검색된 청크를 모델에 보낼 때 바닥글도 함께 들어가 입력 토큰이 늘어나기 때문입니다. 문서가 길고 청크가 많은 코퍼스에서 영향이 커지는지는 이 실험으로 알 수 없습니다.

## 메타데이터

> **요점**
>
> 메타데이터는 청크에 붙이는 문서 번호, 유형, 버전, 날짜, 부서입니다. 임베딩 점수로는 알 수 없는 조건으로 검색 범위를 줄이고, 응답에 출처를 붙일 수 있습니다.

"최신 개정본 기준으로", "품질보증팀 문서만" 같은 조건은 임베딩 점수가 처리하지 못합니다. 유사도는 글의 의미가 비슷한 정도이고, 문서가 최신인지 어느 부서의 것인지는 글의 의미가 아니기 때문입니다. 청크마다 메타데이터를 붙여 두면 이런 조건을 검색 전에 필터링으로 적용할 수 있습니다. 출처를 응답에 적는 데에도 문서 번호와 버전이 필요합니다.

메타데이터는 네 곳에서 뽑습니다.

- **문서 목록 파일**(manifest): 문서마다 유형, 날짜, 버전을 적어 둔 파일입니다(`corpus_manifest.csv`). 가장 확실합니다.
- **파일 이름**: `SOP-ETCH-012_v2.0`에서 유형(SOP)과 버전(v2.0)을 뽑습니다.
- **문서 머리말**: `작성: 설비기술팀`, `관리 부서: 환경안전팀` 같은 줄에서 부서를 뽑습니다.
- **Markdown의 front matter**: 문서 맨 앞의 `---`로 둘러싼 부분입니다. 일부 문서에 있습니다.

메타데이터는 청크를 만들 때 함께 붙여 두면 검색할 때마다 다시 찾지 않아도 됩니다. 출처를 달려면 최소한 문서 번호와 버전이 필요하고, 날짜는 최신 문서를 고를 때 씁니다. 이 파트의 `process()`는 문서 번호, 유형, 버전, 부서를 붙이고 날짜는 붙이지 않습니다. 날짜는 문서 목록 파일에서 가져올 수 있습니다.

**예제 2-6. 메타데이터를 붙이고 필터링하기**

In [ ]:
# 예제 2-6
import csv

META = {row['file'][:-3]: row for row in csv.DictReader(open('data/2일차/corpus_manifest.csv', encoding='utf-8-sig'))}
print(META['SOP-ETCH-012_v2.0'])

TYPES = r'(SOP|QIR|SPEC|MTG|NOTI|EHS|MAN|EDU)'


def type_of(name):
    """파일 이름 앞부분에서 문서 유형을 뽑습니다. 규칙에 맞지 않으면 '-'."""
    found = re.match(TYPES + '-', name)
    return found[1] if found else '-'


def version_of(name):
    """파일 이름 끝의 _v2.0에서 버전을 뽑습니다. 없으면 v1.0."""
    found = re.search(r'_v([\d.]+)$', name)
    return f'v{found[1]}' if found else 'v1.0'


print(type_of('SOP-ETCH-012_v2.0'), version_of('SOP-ETCH-012_v2.0'), type_of('장비마스터'), version_of('SPEC-002'))

{'doc_id': 'SOP-ETCH-012', 'type': 'SOP', 'title': '에치 He 배면압 설정 기준 (개정판)', 'equip': 'ETCH 공통', 'date': '2026-03-12', 'version': 'v2.0', 'file': 'SOP-ETCH-012_v2.0.md'}
SOP v2.0 - v1.0


문서 머리말에서 부서를 뽑습니다. 부서가 없는 문서는 `-`로 두고, 두 부서가 `/`로 붙은 값은 부서마다 나눕니다.

In [ ]:
# 예제 2-6 (부서)
def dept_of(text):
    """머리말의 '작성', '관리 부서', '게시 부서' 줄에서 부서를 뽑습니다. 없으면 '-'."""
    found = re.search(r'(?:작성|관리 부서|게시 부서): *([^\s·]+)', text)
    return found[1] if found else '-'


def split_depts(value):
    return value.split('/') if value != '-' else []


DEPT = {name: dept_of(doc) for name, doc in zip(names, docs)}
print(Counter(DEPT.values()).most_common(8))
joined = [name for name in names if '/' in DEPT[name]]
print(f'{len(joined)}건:', DEPT[joined[0]])

[('-', 73), ('품질보증팀', 59), ('총무팀/설비기술팀', 30), ('공정기술그룹', 30), ('설비기술팀', 26), ('환경안전팀', 20), ('인재개발팀', 15)]
30건: 총무팀/설비기술팀


청크마다 메타데이터를 붙여 놓고 필터링합니다.

In [ ]:
# 예제 2-6 (필터링)
chunks, owners = [], []
for name, doc in zip(names, docs):
    for chunk in chunk_sections(doc):
        chunks.append(chunk)
        owners.append(name)
print('청크', len(chunks))

keep = [position for position, owner in enumerate(owners) if META[owner]['type'] != '회의록']
print('회의록 제외', len(chunks), '→', len(keep))
quality = [position for position, owner in enumerate(owners) if DEPT[owner] == '품질보증팀' and META[owner]['date'] >= '2026-01-01']
print('품질보증팀, 2026년 이후', len(quality))
plain_match = [position for position, owner in enumerate(owners) if DEPT[owner] == '설비기술팀']
split_match = [position for position, owner in enumerate(owners) if '설비기술팀' in split_depts(DEPT[owner])]
print('설비기술팀 문서의 청크', len(plain_match), '→', len(split_match))

청크 757
회의록 제외 757 → 613
품질보증팀, 2026년 이후 100
설비기술팀 문서의 청크 102 → 132


Markdown 문서 맨 앞의 front matter도 읽습니다.

In [ ]:
# 예제 2-6 (front matter)
def front_matter(text):
    """문서 맨 앞의 --- 로 둘러싼 부분을 (dict, 나머지 본문)으로 나눕니다. 없으면 ({}, 전체)."""
    if not text.startswith('---\n'):
        return {}, text
    head, _, body = text[4:].partition('\n---\n')
    pairs = [line.split(': ', 1) for line in head.split('\n') if ': ' in line]
    return {key: value for key, value in pairs}, body


meta, body = front_matter(open(f'{DATA}/docs_raw_m2/md/SPEC-031.md', encoding='utf-8').read())
print(meta)
print(repr(body[:30]))

{'doc_id': 'SPEC-031', 'type': '공정스펙', 'equip': 'ETCH-04', 'date': '2026-08-22', 'version': 'v1.0'}
'# SPEC-031 RCP-ETCH-812 공정 파라미'


문서 목록 파일은 문서 한 건에 대해 `doc_id`, `type`, `title`, `equip`, `date`, `version`, `file`을 줍니다. 파일 이름에서도 유형과 버전을 뽑을 수 있고, 파일 이름이 규칙에 맞지 않으면(`장비마스터`) `-`와 `v1.0`을 씁니다. 버전이 없는 파일 이름에 `v1.0`을 쓰는 것은 문서 목록 파일이 버전 없는 문서를 `v1.0`으로 적은 규칙을 따른 것입니다. 버전이 정해지지 않은 문서라면 `-`로 두는 편이 정확합니다. 파일 이름에서 뽑은 유형(`MTG`)과 문서 목록 파일의 유형(`회의록`)은 표기가 다르므로, 필터링에는 한쪽 표기를 정해 씁니다. 이 파트의 필터링은 문서 목록 파일의 표기를 씁니다.

머리말에서 부서를 뽑으면 부서가 없는 문서가 73건(회의록 48건, 매뉴얼 25건)입니다. 이 문서들은 `-`로 두고 목록으로 따로 남겨, 부서로 필터링하면 빠진다는 것을 알 수 있게 합니다. 공지 30건은 부서가 `총무팀/설비기술팀`처럼 두 개가 `/`로 붙어 있습니다. 이 값을 그대로 비교하면 "설비기술팀 문서"를 찾는 필터에서 이 30건의 청크가 빠집니다. 부서를 `/`로 나눠 비교하면 설비기술팀의 청크가 102건에서 132건이 됩니다. 같은 뜻의 값이 여러 표기로 적혀 있으면 필터링 전에 하나로 정리합니다.

필터링의 효과는 검색 대상인 청크 수의 감소로 보입니다. 회의록을 제외하면 청크가 757건에서 613건으로 줄고(19.0% 감소), 품질보증팀의 2026년 이후 문서의 청크는 100건입니다. 회의록은 "무슨 이야기가 오갔는가"를 찾을 때는 쓸모가 있지만 "기준이 무엇인가"를 찾을 때는 필요 없는 후보입니다. 검색 결과가 좋아지는지는 이 예제에서 측정하지 않았습니다. 날짜를 문자열로 비교하는 `>= '2026-01-01'`은 날짜가 `YYYY-MM-DD` 꼴일 때만 맞습니다.

Markdown의 front matter는 `doc_id`, `type`, `equip`, `date`, `version`을 줍니다. 이 정보는 본문이 아니므로 읽어서 메타데이터로 쓰고 본문에서는 뺍니다. 빼지 않으면 `doc_id: SPEC-031` 같은 줄이 청크에 섞입니다. 예제 2-7의 `process()`도 Markdown 파일은 이 함수로 front matter를 뺀 본문을 씁니다.

## 파이프라인으로 묶기

> **요점**
>
> 형식별 파서, 정리, 청킹, 메타데이터를 함수 하나로 묶고, 처리 보고를 남깁니다. 열리지 않는 파일을 실패 목록에 남기지 않으면 검색이 안 되는 이유를 찾을 수 없습니다.

지금까지 만든 부분을 함수 하나로 묶습니다. 파서는 형식별로 따로 있고, 청킹과 메타데이터는 형식과 관계없이 같은 코드를 씁니다. 정리의 노이즈 규칙은 엑셀을 뺀 형식에 적용합니다. 엑셀의 행 문장은 표의 데이터이기 때문입니다. 이 함수의 청킹은 글자 수(400자)로만 자르고, DOCX의 `Heading 2` 같은 구조 정보는 쓰지 않습니다.

파이프라인이 반드시 남겨야 하는 것은 처리 보고입니다. 어떤 문서가 인덱스에 들어가지 못했는지 알 수 없으면, "왜 이 문서는 검색이 안 되지"의 원인을 찾을 수 없기 때문입니다. 처리 보고에는 다음을 적습니다.

- 열린 파일 수: 파이프라인이 실제로 처리한 파일의 수입니다.
- 지운 줄 수: 정리 단계가 지운 줄의 수입니다. 갑자기 늘거나 줄면 문서 양식이나 노이즈 규칙이 바뀐 것일 수 있습니다.
- 열리지 않은 파일: 파일 이름과 오류의 이름입니다. 파일 한 건이 열리지 않아도 나머지는 계속 처리합니다.

문서 처리를 하다 흔히 만나는 오류는 다음과 같습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `UnicodeDecodeError` | 파일이 UTF-8이 아닙니다(한국어 Windows의 `cp949` 등). `encoding='cp949'`로 엽니다 |
| `EmptyFileError` | 0바이트 PDF 파일입니다(`pypdf`가 냅니다). 전송이 중간에 끊겼는지 확인합니다. 0바이트 DOCX와 XLSX는 아래의 `BadZipFile`이나 `ValueError`로 나옵니다 |
| `BadZipFile` | DOCX와 XLSX는 XML 파일 여러 개를 ZIP으로 묶은 형식입니다. 확장자는 `docx`나 `xlsx`인데 ZIP 파일이 아니면 `python-docx`와 `openpyxl.load_workbook`이 이 오류를 발생시킵니다. 0바이트 파일이거나 다른 형식으로 저장된 파일일 수 있습니다 |
| `PackageNotFoundError` | `python-docx`에 경로를 문자열로 넘겼는데 파일이 ZIP 파일이 아닙니다. `BadZipFile`과 원인이 같고, 경로를 `Path` 객체로 넘기면 `BadZipFile`로 나옵니다 |
| `ValueError: Excel file format cannot be determined` | `pd.read_excel`이 XLSX(ZIP 파일)로 읽지 못한 파일입니다. `BadZipFile`과 원인이 같습니다 |
| `ValueError: 글자 없음` | 글자가 0자로 나온 파일입니다. 스캔한 PDF라면 글자 인식(OCR)이 필요한 파일로 따로 모읍니다 |

**예제 2-7. 파이프라인 함수와 처리 보고**

In [ ]:
# 예제 2-7
from pathlib import Path

SKIP_ROWS = {'공정스펙_관리한계_일람': 3, '장비마스터': 0}      # 장식 줄의 수가 파일마다 달라 파일 이름별로 적습니다


def read_xlsx(path):
    """엑셀의 각 행을 '열 이름 값, 열 이름 값' 한 줄로 만들어 줄바꿈으로 이어 돌려줍니다."""
    table = pd.read_excel(path, skiprows=SKIP_ROWS[Path(path).stem])
    return '\n'.join(', '.join(f'{column} {value}' for column, value in row.items() if pd.notna(value))
                     for _, row in table.iterrows())


def read_file(path):
    """형식에 맞는 도구로 파일을 열어 글을 돌려줍니다. 열리지 않거나 글자가 없으면 오류를 발생시킵니다."""
    suffix = Path(path).suffix.lower()
    if suffix == '.pdf':
        text = '\n'.join(page.extract_text() for page in PdfReader(path).pages)
    elif suffix == '.docx':
        text = '\n'.join(paragraph.text for paragraph in Document(path).paragraphs)
    elif suffix == '.xlsx':
        text = read_xlsx(path)
    else:
        _, text = front_matter(open(path, encoding='utf-8').read())   # front matter는 본문이 아니므로 뺍니다(예제 2-6)
    if not text.strip():
        raise ValueError('글자 없음')                    # 스캔한 PDF처럼 글자가 0자인 파일도 실패 목록에 남깁니다
    return text


def process(root, size=400):
    """폴더 아래의 모든 파일을 열어 (청크 목록, 처리 보고)를 돌려줍니다."""
    chunks, report = [], {'열린 파일': 0, '지운 줄': 0, '실패': []}
    for path in sorted(Path(root).rglob('*')):
        if not path.is_file():
            continue
        try:
            text = read_file(path)
        except Exception as error:                       # 열리지 않은 파일은 목록에 남기고 계속합니다
            report['실패'].append((path.name, type(error).__name__))
            continue
        lines = [CONTROL.sub('', line).strip() for line in text.split('\n') if line.strip()]
        if path.suffix.lower() != '.xlsx':                # 엑셀의 행 문장은 표의 데이터이므로 노이즈 규칙을 적용하지 않습니다
            kept = [line for line in lines if not is_noise(line)]
        else:
            kept = lines
        report['열린 파일'] += 1
        report['지운 줄'] += len(lines) - len(kept)
        for piece in chunk_fixed('\n'.join(kept), size):
            chunks.append({'doc_id': path.stem, 'type': type_of(path.stem), 'version': version_of(path.stem),
                           'dept': dept_of('\n'.join(kept)), 'source': path.name, 'text': piece})
    return chunks, report

예제의 원본 문서 20건을 처리합니다.

In [ ]:
# 예제 2-7 (원본 문서 20건)
chunks, report = process(RAW)
print(report)
print(f'{len(chunks)}청크,', Counter(chunk['type'] for chunk in chunks))
print(chunks[0])

{'열린 파일': 20, '지운 줄': 32, '실패': []}
30청크, Counter({'MTG': 10, 'SOP': 10, '-': 10})
{'doc_id': 'MTG-2025-W01', 'type': 'MTG', 'version': 'v1.0', 'dept': '-', 'source': 'MTG-2025-W01.docx', 'text': 'MTG-2025-W01 장비팀 주간회의록 (2025-08-02)\n참석: 파트장 1, 설비 엔지니어 5, 품질 담당 1\n등급: 사내 일반\n논의 사항\n케이블 트레이 정리 미흡이 안전점검에서 지적되어 주중 재정리하기로 했습니다.\nCVD-03 PM 이 부품 수급 지연으로 13일 밀려 금요일로 재조정되었습니다.\nPHOTO-02 진공펌프 소음 보고가 10건 접수되어 다음 PM 때 베어링을 점검하기로 했습니다.\n스크랩 사유 코드 정리 요청이 품질팀에서 접수되었습니다 (기한 금요일).\nETCH-03 시즈닝 런 파티클이 기준을 초과해 재세정 후 재실시하기로 했습니다.\n결정 사항\n안건별 담당자는 회의 종료 후 파트장이 개별 통지했습니다.'}


열리지 않는 파일이 든 폴더도 처리합니다.

In [ ]:
# 예제 2-7 (열리지 않는 파일)
_, report = process(f'{DATA}/docs_raw_m2_ext')
print(report)

{'열린 파일': 0, '지운 줄': 0, '실패': [('EHS-022.docx', 'BadZipFile'), ('NOTI-032.pdf', 'EmptyFileError')]}


노이즈 규칙을 엑셀에도 적용하면 어떻게 되는지도 봅니다.

In [ ]:
# 예제 2-7 (노이즈 규칙을 엑셀에도 적용하면)
spec_lines = read_xlsx(f'{RAW}/xlsx/공정스펙_관리한계_일람.xlsx').split('\n')
print(f'{len(spec_lines)}줄 중 노이즈로 판정된 줄 {sum(is_noise(line) for line in spec_lines)}줄')

30줄 중 노이즈로 판정된 줄 30줄


원본 문서 20건이 모두 열렸고 정리 단계가 지운 줄은 32줄입니다. PDF 8건에서 4줄씩 지운 수입니다. 청크는 30건이고, 유형별로는 회의록(`MTG`) 10건, SOP 10건, 유형을 알 수 없는 엑셀 파일의 청크 10건입니다. 첫 청크의 `dept`가 `-`인 것은 회의록에 부서 줄이 없기 때문입니다.

열리지 않는 파일 두 건은 실패 목록에 `('EHS-022.docx', 'BadZipFile')`와 `('NOTI-032.pdf', 'EmptyFileError')`로 남습니다. 이 폴더에서는 열린 파일이 0건인데도 함수가 오류 없이 끝납니다. 처리 보고를 읽어야 이 사실을 알 수 있습니다. 한 달 뒤 누군가 이 두 문서가 왜 검색되지 않느냐고 물으면, 실패 목록이 그 질문의 답입니다.

노이즈 규칙을 적용하는 범위도 처리 보고로 점검합니다. 엑셀의 행 문장에는 열 이름 `문서번호`가 들어 있어서, 노이즈 낱말 목록의 `문서번호`가 공정 스펙의 30행을 모두 노이즈로 판정합니다(마지막 출력에서 30줄 중 30줄). 그래서 `process()`는 엑셀의 행 문장에 노이즈 규칙을 적용하지 않습니다. 모든 형식에 적용하면 지운 줄이 32줄이 아니라 62줄이 되고 공정 스펙의 청크가 사라지는데, 오류는 나지 않습니다. 지운 줄 수가 예상보다 크게 늘면 규칙이 본문까지 지우고 있는 것입니다. 처리 보고에 지운 줄 수를 남기는 이유입니다.

이 함수는 엑셀의 건너뛸 줄 수를 `SKIP_ROWS`에 파일마다 적어 두었습니다. 엑셀 파일이 많아지면 이 목록을 손으로 관리하기 어렵습니다. 열 이름 줄을 코드로 찾는 방법은 실습 2에서 작성합니다.

## 실습

실습은 `3일차_02_실습.ipynb`에서 합니다. 임베딩 모델을 PC에서 실행하고, 준비 셀의 연결 확인 한 번 외에는 API를 호출하지 않습니다. 풀이와 해설은 `3일차_02_실습_풀이.ipynb`에 있습니다.

**실습 2. 새 원본 문서를 청크로 만드는 함수 작성하기**

처음 보는 원본 문서 12건(`docs_raw_m2/` 10건과 `docs_raw_m2_ext/` 2건)을 처리하는 함수를 직접 작성합니다. 반복 줄의 후보를 읽고 지울 줄을 판단하고, 엑셀의 열 이름 줄을 찾는 `find_header_row()`, 엑셀을 문장으로 푸는 `read_xlsx()`, 노이즈를 지우는 `drop_noise()`, 폴더를 처리하는 `process()`를 작성합니다. 처리한 청크로 청크 크기 100자와 400자의 recall@1, recall@3을 측정합니다.

완료 기준: 테스트 셀이 "실습 2 통과"를 출력합니다. 지운 줄 수와 실패 목록, 크기별 측정 숫자를 적고, 이 숫자만으로 청크 크기를 고를 수 있는지 한 줄로 판단을 적습니다.

## 핵심 정리

- 문서 처리는 파싱, 정리, 청킹, 메타데이터 붙이기의 네 단계이고, 임베딩 모델은 그 결과인 청크의 글만 봅니다. 검색이 이상하면 이 단계의 결과를 먼저 읽습니다.
- PDF는 구조 정보가 없고 머리말과 표가 섞입니다. 제목 스타일을 쓴 DOCX는 구조를 그대로 쓸 수 있고, XLSX는 건너뛸 장식 줄의 수가 파일마다 다릅니다. 원본이 DOCX로 남아 있으면 PDF 대신 DOCX를 씁니다.
- 반복 줄 탐지는 후보를 찾는 도구입니다. 같은 양식의 문서에서는 제목과 정형 문장도 후보에 들어오므로 지우기 전에 사람이 읽습니다.
- 표를 통째로 임베딩하면 최대 입력 길이를 넘는 뒷부분이 벡터에 반영되지 않습니다. 행을 열 이름이 붙은 한 문장으로 풀어 넣습니다.
- 청크 크기는 권장값이 아니라 질문 세트로 측정해 정하고, 질문이 적으면 크기의 순서가 바뀔 수 있습니다. 중첩은 문서가 청크보다 여러 배 길 때 청크 수를 늘립니다.
- 메타데이터는 청크를 만들 때 붙이고, 같은 뜻의 값은 하나로 정리합니다. 열리지 않은 파일은 처리 보고의 실패 목록에 남깁니다.

## 자가점검

**1.** 같은 회의록이 PDF와 DOCX로 모두 남아 있습니다. 어느 쪽을 파싱해야 하고, 그 이유는 무엇입니까?

<details><summary>정답 보기</summary>

DOCX를 파싱합니다. 작성자가 제목 스타일(`Heading 1`, `Heading 2`)을 썼다면 제목과 본문이 구분되어, 제목을 추측하는 규칙 없이 절 단위로 자를 수 있습니다. 스타일이 `Normal`뿐이면 DOCX도 제목을 추측해야 합니다. PDF에는 글자와 위치만 있어서 제목을 규칙으로 추측해야 합니다.

</details>

**2.** 전체 문서의 80%를 넘는 문서에 나오는 줄을 세어 반복 줄을 찾았더니 회의록 10건에서 "논의 사항", "결정 사항"이 후보로 나왔습니다. 후보를 모두 지워도 됩니까?

- A. 지워도 된다. 80%를 넘는 문서에 나오는 줄은 확실히 머리말이나 바닥글이다
- B. 지우면 안 된다. 같은 양식의 문서에서는 절 제목과 정형 문장도 반복되므로 사람이 후보를 읽고 지울 줄을 정한다
- C. 80%를 90%로 올리면 후보가 정확해지므로 그때는 지워도 된다

<details><summary>정답 보기</summary>

**B.** 회의록은 양식이 같아서 절 제목과 정형 문장이 문서마다 똑같이 나옵니다. 반복 줄 탐지는 후보를 찾는 도구이고, 지울 줄은 사람이 읽고 정합니다(예제 2-2).

</details>

**3.** 모든 문서에 같은 머리말과 바닥글이 남아 있는 채로 임베딩하면 검색에 어떤 일이 생깁니까?

<details><summary>정답 보기</summary>

청크끼리의 유사도 평균이 올라갑니다(분할 없음 0.865에서 0.892). 글자 수로 자른 경우에는 청크 수도 늘어납니다(글자 100은 825건에서 990건). 이 코퍼스에서 질문 20개의 recall@3은 달라지지 않았고, 점수 차도 눈에 띄게 줄지 않았습니다(분할 없음 0.0226에서 0.0210). 순위를 해치지는 않았지만, 문서마다 같은 글은 청크를 구별하는 정보가 아니고 청크 수와 입력 토큰만 늘어나므로 지웁니다(예제 2-5).

</details>

**4.** 30행짜리 표를 통째로 임베딩했더니 마지막 행을 묻는 질문에서 그 행이 검색되지 않습니다. 원인과 해결 방법은 무엇입니까?

<details><summary>정답 보기</summary>

표 전체가 722토큰이어서 임베딩 모델의 최대 입력 길이 512토큰을 넘기 때문입니다. 넘는 뒷부분은 벡터에 반영되지 않았고, 임베딩 결과에는 오류도 경고도 나타나지 않았습니다. 행마다 열 이름을 붙여 한 문장으로 풀면 행마다 벡터가 생겨, 질문이 각 행과 직접 비교됩니다(예제 2-3).

</details>

**5.** 질문 10개로 측정했더니 100자 청크의 recall@3이 60%, 400자 청크가 80%였습니다. 이 결과만으로 "청크는 클수록 좋다"고 결론 내려도 됩니까?

<details><summary>정답 보기</summary>

안 됩니다. 같은 인덱스를 질문 20개로 측정하면 글자 100의 recall@3이 95%로 가장 높아 순서가 달라집니다(예제 2-5). 두 질문 세트 모두 가장 높은 값과 낮은 값의 차이가 질문 두 개 수준이어서, 질문 세트가 달라지면 크기의 순서가 바뀝니다. 이 코퍼스는 문서가 짧아(중앙값 272자) 크기를 바꿔도 청크가 크게 달라지지 않습니다. 청크 크기는 내 문서와 질문 세트로 측정해 정하고, 질문이 열 개 안팎이면 결론을 내리지 않습니다.

</details>

**6.** 청크에 중첩을 주면 어떤 일이 일어나고 어떤 비용이 듭니까?

- A. 경계에 걸친 문장이 잘리는 문제를 줄이지만 청크 수와 인덱스 크기가 늘어난다
- B. 청크 수와 인덱스 크기는 그대로이고 검색 점수만 올라간다
- C. 문서가 짧아 청크가 한두 개뿐일 때 효과가 가장 크다

<details><summary>정답 보기</summary>

**A.** 앞 청크의 끝부분을 다음 청크의 앞에도 넣으므로, 문서가 청크보다 여러 배 길면 청크 수가 늘어납니다. 200자 청크에 중첩 60자를 주면 청크가 469건에서 526건(1.12배)이 됩니다(예제 2-5). 문서가 짧아 청크가 한두 개뿐이면 경계가 적어 효과가 작고, 400자 청크에서는 청크 수도 282건 그대로였습니다(예제 2-4).

</details>

**7.** 청크에 문서 유형, 버전, 날짜, 부서 같은 메타데이터를 붙이는 이유는 무엇입니까? 부서 값이 `총무팀/설비기술팀`처럼 적힌 문서가 있을 때 주의할 점은 무엇입니까?

<details><summary>정답 보기</summary>

임베딩 점수로는 알 수 없는 조건(유형, 최신 여부, 부서)으로 검색 범위를 필터링하고, 응답에 출처를 붙이기 위해서입니다. `총무팀/설비기술팀`을 그대로 비교하면 "설비기술팀" 필터에서 그 문서의 청크가 빠지므로, 필터링 전에 `/`로 나눠 값을 정리해야 합니다(예제 2-6).

</details>

**8.** 문서 묶음에서 한 파일이 열리지 않았는데 오류 메시지가 나오지 않았고, 한 달 뒤 "그 문서는 왜 검색이 안 되느냐"는 문의가 왔습니다. 파이프라인을 어떻게 고쳐야 합니까?

<details><summary>정답 보기</summary>

파일이 열리지 않아도 나머지를 계속 처리하되, 열리지 않은 파일의 이름과 오류의 이름을 처리 보고의 실패 목록에 남깁니다. 열린 파일 수와 지운 줄 수도 함께 기록하면 어떤 문서가 인덱스에 들어가지 못했는지 바로 알 수 있습니다(예제 2-7).

</details>